# GraphForge — Rejection-Sampling SFT on a free Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nithin062006/scaler/blob/main/training/notebook.ipynb)

This notebook runs the full pipeline end-to-end against the GraphForge OpenEnv environment:

1. Clone the repo and install deps (1 min on T4)
2. Baseline-eval Qwen2.5-0.5B-Instruct against the tier-0 task
3. Generate trajectories (oracle + live model) and reject-sample
4. SFT the kept trajectories (TRL SFTTrainer + LoRA)
5. Trained-eval the same model and write all hackathon plots

Expected wall-clock: ~10–20 min on a T4.

## 1. Setup

In [ ]:
import os, subprocess, pathlib

REPO_URL = 'https://github.com/nithin062006/scaler.git'
cwd = pathlib.Path(os.getcwd())

# Idempotent: handles fresh runtime, restarted runtime, and re-runs.
if (cwd / 'graphforge').exists() and (cwd / 'env').exists():
    print(f'Already inside repo: {cwd}')
elif (cwd / 'graphforge_repo').exists():
    os.chdir('graphforge_repo')
    print(f'Cd-ed into existing clone: {os.getcwd()}')
else:
    subprocess.check_call(['git', 'clone', '-q', REPO_URL, 'graphforge_repo'])
    os.chdir('graphforge_repo')
    print(f'Cloned + cd-ed: {os.getcwd()}')

print(os.listdir('.'))

In [ ]:
# Install runtime + training deps. Pin TRL to a known-stable version to
# avoid the SFTConfig API churn we hit before. peft is needed for LoRA.
%pip install -q -e ".[training]"
%pip install -q "trl==0.11.4" "peft>=0.10,<0.13"
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Patch HfPolicy.sample for newer transformers + post-SFT eval correctness.
# Idempotent — overwrites the file every run.
import pathlib, sys

_FIXED = '''"""Policy interface and stub policies."""

from __future__ import annotations
from typing import Iterator, Protocol, runtime_checkable
from graphforge.training.prompt import Message


@runtime_checkable
class Policy(Protocol):
    def sample(self, messages: list[Message]) -> str: ...


class ScriptedPolicy:
    def __init__(self, completions):
        self._iter = iter(completions)
    def sample(self, _messages):
        return next(self._iter)


class HfPolicy:
    def __init__(self, model, tokenizer, *, max_new_tokens=384, temperature=0.7, top_p=0.95):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p

    def sample(self, messages):
        import torch
        # Critical for trained-eval correctness:
        self.model.eval()
        if hasattr(self.model, "config"):
            self.model.config.use_cache = True
        tok = self.tokenizer
        text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = tok(text, return_tensors="pt")
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,                # GREEDY — deterministic
                pad_token_id=tok.eos_token_id,
                use_cache=True,
            )
        prompt_len = inputs["input_ids"].shape[-1]
        gen = out_ids[0, prompt_len:]
        return tok.decode(gen, skip_special_tokens=True)
'''
pathlib.Path('graphforge/training/policy.py').write_text(_FIXED)

# Drop cached graphforge imports.
_dropped = [k for k in list(sys.modules) if k.startswith('graphforge')]
for k in _dropped:
    del sys.modules[k]
print(f"Patched policy.py (greedy + eval mode); cleared {len(_dropped)} cached imports.")

In [ ]:
# Rewrite training/train.py's _run_sft with a clean, TRL-0.11-compatible
# version that:
#   - uses dataset_text_field + max_seq_length on SFTTrainer (TRL 0.11 idiom)
#   - saves a checkpoint every epoch so eval can reload it
#   - merges LoRA weights back into the base model so the in-memory model
#     used by HfPolicy reflects the trained weights
#   - returns loss_history + steps for plotting
import pathlib, sys

train_path = pathlib.Path('training/train.py')
src = train_path.read_text()

# Slice out the existing _run_sft and replace it wholesale.
start_marker = '\ndef _run_sft('
end_marker   = '\ndef '   # next top-level def
start_idx = src.index(start_marker)
search_from = start_idx + len(start_marker)
end_idx = src.index(end_marker, search_from)

NEW_RUN_SFT = '''
def _run_sft(cfg, model, tok, examples):
    """LoRA SFT with TRL 0.11.x. Saves a checkpoint each epoch."""
    import datasets, torch
    from trl import SFTTrainer
    from transformers import TrainingArguments

    rows = [{"text": ex["prompt"] + ex["completion"]} for ex in examples]
    ds = datasets.Dataset.from_list(rows)
    output_dir = str(cfg.out_dir / "sft")

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        fp16=torch.cuda.is_available(),
        logging_steps=5,
        save_strategy="epoch",        # ← so eval can reload
        save_total_limit=1,
        report_to="none",
        seed=cfg.seed,
    )

    lora_cfg = None
    if cfg.use_lora:
        from peft import LoraConfig, TaskType
        lora_cfg = LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            task_type=TaskType.CAUSAL_LM,
            bias="none",
        )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=ds,
        tokenizer=tok,
        dataset_text_field="text",
        max_seq_length=512,
        peft_config=lora_cfg,
    )

    train_result = trainer.train()

    # Merge LoRA so the in-memory model has trained weights.
    inner = trainer.model
    if cfg.use_lora and hasattr(inner, "merge_and_unload"):
        try:
            inner.merge_and_unload()
        except Exception as e:
            print(f"[train] LoRA merge warning (non-fatal): {e}")
    inner.eval()
    if hasattr(inner, "config"):
        inner.config.use_cache = True

    # Extract per-step losses for plotting.
    log_history = trainer.state.log_history
    steps  = [e["step"] for e in log_history if "loss" in e]
    losses = [e["loss"] for e in log_history if "loss" in e]
    return {
        "loss_history":  losses,
        "steps":         steps,
        "train_loss":    train_result.training_loss,
        "train_runtime": train_result.metrics.get("train_runtime", 0),
    }

'''

src = src[:start_idx] + NEW_RUN_SFT + src[end_idx:]
train_path.write_text(src)
print("✓ rewrote _run_sft (TRL 0.11.x compatible, saves epoch checkpoints, merges LoRA)")

# Drop cached imports.
_dropped = [k for k in list(sys.modules) if k.startswith('graphforge') or k.startswith('training')]
for k in _dropped:
    del sys.modules[k]
print(f"✓ cleared {len(_dropped)} cached imports")

## 2. Run training

`training.train.run` does baseline eval → trajectory generation → SFT (LoRA) → trained eval (in-pipeline, approximate) → plots. The comprehensive eval below uses the saved checkpoint to do a clean before/after with identical greedy decoding.

In [ ]:
# Shared MODEL_NAME — used by both the train cell and the eval cell so they
# always match. Change this once if you want a different base model.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

from pathlib import Path
from training.config import TrainConfig
from training.train import run

cfg = TrainConfig(
    model_name=MODEL_NAME,
    task_id='t0.email_validator',
    max_new_tokens=64,
    episode_cap=8,
    n_oracle=20,
    n_explore=2,
    reward_threshold=5.0,
    epochs=10,                # ← bump this for more training
    learning_rate=2e-4,
    batch_size=4,             # 4×grad_accum=2 → effective batch 8 (fits 0.5B on T4)
    gradient_accumulation_steps=2,
    use_lora=True,            # LoRA — works reliably with TRL 0.11
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    n_eval_episodes=3,        # in-pipeline eval is informational; real eval is below
    out_dir=Path('outputs'),
    plots_dir=Path('plots'),
)
summary = run(cfg)
print('=' * 60)
print(f"in-pipeline baseline mean = {summary['baseline_eval']['mean_reward']:+.2f}")
print(f"in-pipeline trained mean  = {summary['trained_eval']['mean_reward']:+.2f}")
print('=' * 60)
print("(in-pipeline trained eval is approximate — the comprehensive eval cell")
print(" below is the real before/after measurement)")

## 3. Comprehensive evaluation

Loads the saved checkpoint, evaluates baseline AND trained models with the **same** generation settings (greedy + truncate at first `</action>`) so the comparison is apples-to-apples. Writes the canonical `outputs/baseline_eval.json` + `outputs/trained_eval.json` that the stats cell consumes.

## 4. Stats table + all plots inline

## 5. Commit the artifacts

When you're happy with the numbers and plots, commit `plots/*.png` and `outputs/*.json` back to the repo:

```bash
git add plots/ outputs/baseline_eval.json outputs/trained_eval.json
git commit -m "Real Colab training run + before/after eval"
git push
```

Then deploy the env to a Hugging Face Space and add the URL to the README. Submit.

## 5. Commit the artifacts

When you're happy with the numbers and plots, commit `plots/*.png` and `outputs/*.json` back to the repo:

```bash
git add plots/ outputs/baseline_eval.json outputs/trained_eval.json
git commit -m "Real Colab training run + before/after eval"
git push
```

Then deploy the env to a Hugging Face Space and add the URL to the README. Submit.

In [ ]:
from IPython.display import Image, display
for name in ['comparison.png', 'baseline_rewards.png', 'trained_rewards.png', 'loss_curve.png']:
    p = Path('plots') / name
    if p.exists():
        print(name)
        display(Image(str(p)))